# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện

> **Track 4 · Ngày 1 · VinUniversity AICB 2026**
> Notebook này chứa toàn bộ pipeline từ Part 0 đến Part 4: chuẩn bị dữ liệu, kiểm tra sức khoẻ mô hình, baseline, các nhóm thí nghiệm, và đánh giá trên tập eval.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
from pathlib import Path
import numpy as np, torch

# Thiết lập đường dẫn linh hoạt (chạy từ submission_<MSSV>/code/ hoặc từ code/)
if os.path.exists("../../data"):
    REPO_ROOT = "../.."
    OUT_DIR = ".."
elif os.path.exists("../data"):
    REPO_ROOT = ".."
    OUT_DIR = "../submission_02594" if os.path.exists("../submission_02594") else "."
else:
    REPO_ROOT = "."
    OUT_DIR = "./submission_02594" if os.path.exists("./submission_02594") else "."

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Thiết bị huấn luyện: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

ModuleNotFoundError: No module named 'torch'

## Part 0 — Dữ liệu
1. Chạy `scripts/split_data.py` nếu chưa có `data/processed/train.npz`.
2. Tách Validation (20%) phân tầng theo nhãn từ Train.
3. Chuẩn hóa chỉ trên 10 cột số liên tục bằng mean/std của Train (tránh data leakage).
4. Đưa toàn bộ tensor lên device.

In [ ]:
proc_dir = f"{REPO_ROOT}/data/processed"
if not os.path.exists(f"{proc_dir}/train.npz"):
    print("Đang chia dữ liệu bằng scripts/split_data.py...")
    subprocess.run([sys.executable, f"{REPO_ROOT}/scripts/split_data.py",
                    "--data", f"{REPO_ROOT}/data/covtype.csv.gz",
                    "--meta", f"{REPO_ROOT}/data/split_metadata.csv",
                    "--out", proc_dir], check=True)

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=proc_dir)

print("\nKiểm tra thống kê 10 cột số liên tục trên X_tr sau chuẩn hoá:")
print("Mean (kỳ vọng ~0):", data["X_tr"][:, :10].mean(dim=0).cpu().numpy().round(3))
print("Std  (kỳ vọng ~1):", data["X_tr"][:, :10].std(dim=0).cpu().numpy().round(3))

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Kiểm tra 5 tiêu chí bắt buộc:
1. Số lượng tham số `M-base` = 47.879, logits shape `(B, 7)`.
2. Loss bước 0 trên Val $\approx \ln 7 \approx 1.946$.
3. Quá khớp 20 mẫu: loss tiến về ~0, accuracy = 100%.
4. Gradient chảy tới mọi tham số (khác None và > 0).

In [ ]:
# 1. Model M-base shape & param assert
model_check = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model_check)
print(f"1. Số tham số M-base: {n_params} (Kỳ vọng: {EXPECTED_PARAMS[(256, 128)]})")
assert n_params == EXPECTED_PARAMS[(256, 128)], f"Sai số tham số: {n_params}"

# 2. Random batch shape check
dummy_x = torch.randn(8, 54, device=device)
dummy_logits = model_check(dummy_x)
print(f"2. Input shape: {dummy_x.shape} -> Logits shape: {dummy_logits.shape}")
assert dummy_logits.shape == (8, 7), "Logits shape phải là (8, 7)"

# 3. Loss bước 0 trên val ≈ ln 7 ≈ 1.946
step0_eval = evaluate(model_check, data["X_val"], data["y_val"], loss_name="ce")
print(f"3. Loss bước 0 trên Val: {step0_eval['loss']:.4f} (ln 7 = {np.log(7):.4f})")

# 4. Quá khớp 20 mẫu
print("\n4. Đang kiểm tra quá khớp 20 mẫu...")
x_20 = data["X_tr"][:20].clone()
y_20 = data["y_tr"][:20].clone()
m_overfit = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt_overfit = torch.optim.Adam(m_overfit.parameters(), lr=0.01)
for step in range(250):
    opt_overfit.zero_grad()
    loss = torch.nn.functional.cross_entropy(m_overfit(x_20), y_20)
    loss.backward()
    opt_overfit.step()

final_overfit_acc = (m_overfit(x_20).argmax(dim=1) == y_20).float().mean().item()
print(f"   Accuracy sau 250 bước: {final_overfit_acc * 100:.1f}%, Loss: {loss.item():.6f}")
assert final_overfit_acc == 1.0, "Không quá khớp được 20 mẫu!"

# 5. Kiểm tra gradient chảy tới mọi tham số
opt_overfit.zero_grad()
loss = torch.nn.functional.cross_entropy(m_overfit(x_20), y_20)
loss.backward()
print("\n5. Gradient norm từng lớp sau backward:")
for name, p in m_overfit.named_parameters():
    assert p.grad is not None, f"Gradient của {name} bị None!"
    gn = p.grad.norm().item()
    print(f"   {name:25s}: {gn:.6f}")
    assert gn > 0, f"Gradient của {name} bằng 0!"
print("Tất cả kiểm tra Part 1 ĐẠT!")

## Part 2 — Pipeline và Baseline
- Quét nhanh `lr` cho baseline trên Val quanh `[0.01, 0.03, 0.05, 0.1]` để chọn `lr` tốt nhất.
- Chạy Baseline `M-base` (SGD + Momentum 0.9, He init, CE loss, batch 512, 20 epochs) trên **3 seeds** (`seed=1, 2, 3`) để đo độ nhiễu và ngưỡng $2\sigma$.

In [ ]:
# Quét nhanh vài lr cho baseline (3 epoch mỗi giá trị để xem xu hướng)
print("--- Quét nhanh lr cho baseline ---")
candidate_lrs = [0.01, 0.03, 0.05, 0.1]
best_lr = 0.05
best_lr_score = -1.0

for test_lr in candidate_lrs:
    cfg_test = {**DEFAULT_CFG, "lr": test_lr, "epochs": 3, "exp_id": f"lr_probe_{test_lr}"}
    res_test = run_experiment(cfg_test, data)
    f1 = res_test["summary"]["val_macro_f1"]
    print(f"  lr = {test_lr:<5}: val_loss = {res_test['summary']['best_val_loss']:.4f}, val_macro_f1 = {f1:.4f}")
    if f1 > best_lr_score:
        best_lr_score = f1
        best_lr = test_lr

print(f"\n==> Chọn lr baseline = {best_lr}")
base_cfg = {**DEFAULT_CFG, "lr": best_lr, "epochs": 20}

# Chạy baseline trên 3 seeds để đo độ nhiễu seed
baseline_results = []
for s in (1, 2, 3):
    exp_id = f"base-s{s}"
    print(f"\nĐang chạy baseline: {exp_id} (seed={s})...")
    cfg_s = {**base_cfg, "seed": s, "exp_id": exp_id}
    res_s = run_experiment(cfg_s, data)
    save_result(res_s, f"{OUT_DIR}/results")
    plot_run(res_s, f"{OUT_DIR}/figures/{exp_id}.png")
    baseline_results.append(res_s)

base_f1s = [r["summary"]["val_macro_f1"] for r in baseline_results]
base_accs = [r["summary"]["val_acc"] for r in baseline_results]
mean_f1, std_f1 = np.mean(base_f1s), np.std(base_f1s)
noise_threshold = 2 * std_f1

print(f"\nBaseline Macro-F1: {mean_f1:.4f} ± {std_f1:.4f}")
print(f"Baseline Accuracy: {np.mean(base_accs):.4f} ± {np.std(base_accs):.4f}")
print(f"Ngưỡng nhiễu 2σ (Macro-F1): {noise_threshold:.4f}")

## Part 3 — Các nhóm thí nghiệm (7 chủ đề theo menu GUIDE)
Thiết kế so sánh công bằng: **mỗi thí nghiệm chỉ đổi đúng 1 yếu tố** so với baseline, lưu đồ thị và JSON.

In [ ]:
# Danh sách cấu hình thí nghiệm phủ đủ 7 chủ đề
experiments_to_run = [
    # 1. Loss: CE vs MSE
    {"exp_id": "loss-mse", "group": "loss", "description": "Loss MSE thay vì CE", "loss": "mse"},
    
    # 2. Optimizer: Thử SGD thuần, Adam, AdamW
    {"exp_id": "opt-sgd-lr0.05", "group": "optimizer", "description": "SGD không momentum", "optimizer": "sgd", "lr": 0.05},
    {"exp_id": "opt-adam-lr1e-3", "group": "optimizer", "description": "Adam lr=1e-3", "optimizer": "adam", "lr": 0.001},
    {"exp_id": "opt-adam-lr3e-4", "group": "optimizer", "description": "Adam lr=3e-4", "optimizer": "adam", "lr": 0.0003},
    {"exp_id": "opt-adamw-lr1e-3", "group": "optimizer", "description": "AdamW lr=1e-3 wd=0.01", "optimizer": "adamw", "lr": 0.001, "weight_decay": 0.01},
    
    # 3. Hyper-parameters: Batch size & Kiến trúc
    {"exp_id": "hparam-batch128", "group": "hparam", "description": "Batch size nhỏ 128", "batch": 128},
    {"exp_id": "hparam-batch2048", "group": "hparam", "description": "Batch size lớn 2048", "batch": 2048},
    {"exp_id": "hparam-wide", "group": "hparam", "description": "M-wide (512-256-7)", "hidden": (512, 256)},
    {"exp_id": "hparam-deep", "group": "hparam", "description": "M-deep (256-128-64-7)", "hidden": (256, 128, 64)},
    
    # 4. Dropout
    {"exp_id": "drop-0.1", "group": "dropout", "description": "Dropout rate q=0.1", "dropout": 0.1},
    {"exp_id": "drop-0.3", "group": "dropout", "description": "Dropout rate q=0.3", "dropout": 0.3},
    {"exp_id": "drop-0.5", "group": "dropout", "description": "Dropout rate q=0.5", "dropout": 0.5},
    
    # 5. Gradient Clipping
    {"exp_id": "clip-1.0", "group": "clipping", "description": "Clip gradient max_norm=1.0", "clip_norm": 1.0},
    {"exp_id": "clip-highlr-noclip", "group": "clipping", "description": "High LR (0.5) không clip", "lr": 0.5, "clip_norm": None},
    {"exp_id": "clip-highlr-clipped", "group": "clipping", "description": "High LR (0.5) có clip norm=1.0", "lr": 0.5, "clip_norm": 1.0},
    
    # 6. Mixed Precision
    {"exp_id": "amp-fp16", "group": "amp", "description": "Mixed precision FP16", "precision": "fp16"},
    
    # 7. Khởi tạo tham số
    {"exp_id": "init-zeros", "group": "init", "description": "Khởi tạo toàn bộ W=0", "init": "zeros"},
    {"exp_id": "init-normal", "group": "init", "description": "Khởi tạo Normal(0, 0.01^2)", "init": "normal"},
    {"exp_id": "init-xavier", "group": "init", "description": "Khởi tạo Xavier Normal", "init": "xavier"},
]

all_experiment_results = list(baseline_results)
for exp_spec in experiments_to_run:
    exp_cfg = {**base_cfg, **exp_spec}
    print(f"\n>>> Chạy thí nghiệm [{exp_cfg['exp_id']}] (Nhóm: {exp_cfg['group']}): {exp_cfg['description']}")
    res = run_experiment(exp_cfg, data)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{exp_cfg['exp_id']}.png")
    all_experiment_results.append(res)
    
    diff_f1 = res["summary"]["val_macro_f1"] - mean_f1
    beyond_noise = "VƯỢT NHIỄU (+)" if diff_f1 > noise_threshold else ("VƯỢT NHIỄU (-)" if diff_f1 < -noise_threshold else "Không đáng kể")
    print(f"    Val Macro-F1: {res['summary']['val_macro_f1']:.4f} (Chênh lệch: {diff_f1:+.4f}, {beyond_noise})")

### Vẽ ảnh so sánh theo từng nhóm (`compare_<nhóm>.png`)

In [ ]:
groups = set(r["cfg"].get("group") for r in all_experiment_results)
for g in sorted(groups):
    if g == "baseline":
        continue
    grp_results = [r for r in all_experiment_results if r["cfg"].get("group") == g or r["cfg"].get("exp_id") == "base-s1"]
    if len(grp_results) > 1:
        out_img = f"{OUT_DIR}/figures/compare_{g}.png"
        plot_compare(grp_results, "val_macro_f1", out_img, title=f"So sánh Val Macro-F1: Nhóm {g.upper()}")
        print(f"Đã lưu {out_img}")

## Part 4 — Đánh giá cuối trên tập Eval & Xuất kết quả
- Chọn cấu hình tốt nhất hoàn toàn bằng **Val**.
- Dự đoán trên 116.203 mẫu `eval.npz`, xuất `predictions_eval.csv`.
- Chạy `scripts/evaluate.py` sinh `eval_result.json`.
- Xuất toàn bộ kết quả vào `experiments.xlsx`.

In [ ]:
# Tìm cấu hình tốt nhất dựa trên Val Macro-F1
valid_results = [r for r in all_experiment_results if not r["summary"]["diverged"]]
best_res = max(valid_results, key=lambda r: r["summary"]["val_macro_f1"])
print(f"Cấu hình tốt nhất được chọn: [{best_res['cfg']['exp_id']}]")
print(f"Val Macro-F1: {best_res['summary']['val_macro_f1']:.4f}, Val Acc: {best_res['summary']['val_acc']:.4f}")

# 1. Xuất file dự đoán eval cho cấu hình tốt nhất
pred_eval_path = f"{OUT_DIR}/predictions_eval.csv"
final_eval(best_res["cfg"], best_res, data, pred_eval_path)

# 2. Chạy chấm điểm chính thức bằng scripts/evaluate.py
eval_json_path = f"{OUT_DIR}/eval_result.json"
cmd = [
    sys.executable, f"{REPO_ROOT}/scripts/evaluate.py",
    "--pred", pred_eval_path,
    "--data", f"{REPO_ROOT}/data/covtype.csv.gz",
    "--meta", f"{REPO_ROOT}/data/split_metadata.csv",
    "--out", eval_json_path,
]
res_eval_cmd = subprocess.run(cmd, capture_output=True, text=True, check=True)
print(res_eval_cmd.stdout)

with open(eval_json_path, "r", encoding="utf-8") as f:
    eval_result_dict = json.load(f)

# 3. Cũng tính điểm eval cho baseline base-s1 để điền cột eval của baseline
base_res = baseline_results[0]
pred_base_path = f"{OUT_DIR}/predictions_base.csv"
final_eval(base_res["cfg"], base_res, data, pred_base_path)
base_eval_json_path = f"{OUT_DIR}/eval_base_result.json"
subprocess.run([
    sys.executable, f"{REPO_ROOT}/scripts/evaluate.py",
    "--pred", pred_base_path,
    "--data", f"{REPO_ROOT}/data/covtype.csv.gz",
    "--meta", f"{REPO_ROOT}/data/split_metadata.csv",
    "--out", base_eval_json_path,
], check=True)
with open(base_eval_json_path, "r", encoding="utf-8") as f:
    base_eval_dict = json.load(f)
if os.path.exists(pred_base_path): os.remove(pred_base_path)
if os.path.exists(base_eval_json_path): os.remove(base_eval_json_path)

# 4. Điền bảng Excel experiments.xlsx
rows = []
for r in all_experiment_results:
    exp_id = r["cfg"]["exp_id"]
    if exp_id == best_res["cfg"]["exp_id"]:
        ev_sc = {"accuracy": eval_result_dict["accuracy"], "macro_f1": eval_result_dict["macro_f1"]}
        notes = "Cấu hình tốt nhất nộp bài (chọn bằng Val)"
    elif exp_id == "base-s1":
        ev_sc = {"accuracy": base_eval_dict["accuracy"], "macro_f1": base_eval_dict["macro_f1"]}
        notes = "Baseline tham chiếu"
    else:
        ev_sc = None
        notes = ""
    rows.append(to_row(r, eval_scores=ev_sc, notes=notes))

template_xlsx = f"{REPO_ROOT}/templates/experiment_table_template.xlsx"
out_xlsx = f"{OUT_DIR}/experiments.xlsx"
write_xlsx(rows, template_xlsx, out_xlsx)
print(f"\nĐÃ HOÀN TẤT! Toàn bộ kết quả đã xuất ra thư mục {OUT_DIR}/")